# Infer ***nuclear envelope*** from ER marker
--------------

## OBJECTIVES
### <input type="checkbox"/> Infer sub-cellular component #2️: ***nucleus***
Segment ***nuclei*** from the inverse of the ***ER*** mask. Because the organelles used for the composite are cytoplasmic, the nuclei should remain "empty".


### <input type="checkbox"/> Infer sub-cellular component #2: ***nuclear envelope***
Segment the ***nuclear envelope*** by combining the ***ER*** and ***nucleus*** masks. To create an instance segmentation of the nuclear envelope, the nucleus will be dilated and overlayed onto the ER mask. 


---------
## **nuclear envelope workflow**
### summary of steps

➡️ **EXTRACTION**
- **`STEP 1`** - Isolate ER channel

**PRE-PROCESSING**
- **`STEP 2`** - Rescale and smooth image

    - median filter (median size = user input)
    - gaussian filter (sigma = user input)
 
**CORE PROCESSING**
- **`STEP 3`** - Semantic segmentation of the cytoplasm

    - apply MO thresholding method from the Allen Cell [aicssegmentation](https://github.com/AllenCell/aics-segmentation) package (threshold options = user input)
    - fill holes (hole size = user input)
    - remove small objects (object size = user input)
    - filter method = (method = user input)
    
- **`STEP 4`** - Segmentation of nuclei ‘seeds’ from cytoplasm

    - binary opening of inverted aggregate cytoplasm mask (eroded -> dilated)
    - fill nuclei in aggregate cytoplasm mask (hole size = user input)
    - logical **XOR** of the cytoplasm and the filled in cytoplasm resulting in the nucleus and any artifacts from binary opening (erosion -> dilation)
    - remove small objects (object size = user input)
    
- **`STEP 5`** - Filter and dilate nuclei

    - Remove nuclei touching edges
    - Dilate each pixel in every direction (# of pixels = user input)

- **`STEP 6`** - Segment nuclear envelope

    - logical **and** of the dilated nucleus and the ER resulting in nuclear envelope mask
    - erode via 1) binary erosion (# of pixels = user input) or 2) topology perserving thinning (user input)

**EXPORT** ➡️
- **`STEP 7`** - export nuclear envelope labels

---------
## **IMPORTS**

#### &#x1F3C3; **Run code; no user input required**

&#x1F453; **FYI:** This code block loads all of the necessary python packages and functions you will need for this notebook. 

In [20]:
from pathlib import Path
import os
from typing import Union
import tifffile
import pandas as pd
import numpy as np
import napari
from napari.utils.notebook_display import nbscreenshot

from skimage.segmentation import clear_border 
from skimage.morphology import (binary_opening,
                                binary_dilation,
                                binary_erosion)
from skimage.measure import label

from infer_subc.core.file_io import (read_czi_image,
                                     export_inferred_organelle,
                                     list_image_files)
                                    #  sample_input)

                                             
from infer_subc.core.img import *
from infer_subc.organelles import non_linear_cellmask_transform

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## **LOAD AND READ IN IMAGE FOR PROCESSING**

In [21]:
### USER INPUT REQUIRED ###
# If not using the sample data, set cell_type to None
sample_data_type = None

## RAW IMAGE PATH
## For windows put a little r before ""
in_data_path = Path("/Volumes/Meredith/Meredith Microscopy data/SNCA iPSC MSi/Exp1_8_29_25/0829_deconvolution")

# Specify the file type of your raw data 
im_type = ".tiff" 

## SEGMENTATION OUTPUT PATH
out_data_path = Path("/Volumes/Meredith/Meredith Microscopy data/SNCA iPSC MSi/Exp1_8_29_25/106_ne_test")

#### &#x1F3C3; **Run code; no user input required**

In [22]:
# If sample_data_type is set to "astrocyte", then the sample data is used and the directories are set
if sample_data_type != None:
    data_root_path, im_type, in_data_path, out_data_path = sample_input(sample_data_type)

if not Path.exists(out_data_path):
    Path.mkdir(out_data_path)
    print(f"making {out_data_path}")

img_file_list = list_image_files(in_data_path,im_type)
pd.set_option('display.max_colwidth', None)
# pd.set_option('display.max_rows', None)
pd.DataFrame({"Image Name":img_file_list})

,Image Name
0,/Volumes/Meredith/Meredith Microscopy data/SNCA iPSC MSi/Exp1_8_29_25/0829_deconvolution/H9_ALLwell1_img1_Unmixing_cmle.ome.tiff
1,/Volumes/Meredith/Meredith Microscopy data/SNCA iPSC MSi/Exp1_8_29_25/0829_deconvolution/H9_ALLwell1_img23_Unmixing_cmle.ome.tiff
2,/Volumes/Meredith/Meredith Microscopy data/SNCA iPSC MSi/Exp1_8_29_25/0829_deconvolution/H9_ALLwell1_img24_Unmixing_cmle.ome.tiff
3,/Volumes/Meredith/Meredith Microscopy data/SNCA iPSC MSi/Exp1_8_29_25/0829_deconvolution/H9_ALLwell1_img25_Unmixing_cmle.ome.tiff
4,/Volumes/Meredith/Meredith Microscopy data/SNCA iPSC MSi/Exp1_8_29_25/0829_deconvolution/H9_ALLwell1_img2_Unmixing_cmle.ome.tiff
5,/Volumes/Meredith/Meredith Microscopy data/SNCA iPSC MSi/Exp1_8_29_25/0829_deconvolution/H9_ALLwell1_img3_Unmixing_cmle.ome.tiff
6,/Volumes/Meredith/Meredith Microscopy data/SNCA iPSC MSi/Exp1_8_29_25/0829_deconvolution/H9_ALLwell1_img4_Unmixing_cmle.ome.tiff
7,/Volumes/Meredith/Meredith Microscopy data/SNCA iPSC MSi/Exp1_8_29_25/0829_deconvolution/H9_ALLwell1_img5_Unmixing_cmle.ome.tiff
8,/Volumes/Meredith/Meredith Microscopy data/SNCA iPSC MSi/Exp1_8_29_25/0829_deconvolution/H9_ALLwell1_img6_Unmixing_cmle.ome.tiff
9,/Volumes/Meredith/Meredith Microscopy data/SNCA iPSC MSi/Exp1_8_29_25/0829_deconvolution/H9_ALLwell2_img10_Unmixing_cmle.ome.tiff


In [23]:
### USER INPUT REQUIRED ###
# Specify which file you'd like to segment from the img_file_list
test_img_n = 71

In [24]:
test_img_name = img_file_list[test_img_n]

img_data,meta_dict = read_czi_image(test_img_name)

channel_names = meta_dict['name']
img = meta_dict['metadata']['aicsimage']
scale = meta_dict['scale']
channel_axis = meta_dict['channel_axis']

In [25]:
viewer = napari.Viewer()
viewer.add_image(img_data, scale=scale, name=f"RAW IMAGE")

<Image layer 'RAW IMAGE' at 0x3b0e3fbb0>

# ***EXTRACTION prototype - masks_B***


# ***PRE-PROCESSING prototype - masks_B***

## **`STEP 1` - Create composite image**

- determine weight to apply to each channel of the intensity image (w# = user input)
- rescale summed image intensities (rescale = user input)

In [26]:
###################
# INPUT
###################
ER_CH = 6
raw_ER = select_channel_from_raw(img_data, ER_CH)
viewer.add_image(raw_ER, scale=scale, name=f"ER channel")

<Image layer 'ER channel' at 0x3b0dde770>

## **`STEP 2` - Rescale and smooth image**

- rescale intensity of composite image (min=0, max=1)
- median filter (median size = user input)
- gaussian filter (sigma = user input)

> **NOTE**: No smoothing was done here because these test images were already pre-processed.

In [27]:
### USER INPUT REQUIRED ###
med_filter_size = 0
gaussian_smoothing_sigma = 1.34

structure_img_smooth = scale_and_smooth(raw_ER,
                                        median_size = med_filter_size, 
                                        gauss_sigma = gaussian_smoothing_sigma)

In [ ]:
viewer.add_image(structure_img_smooth, scale=scale, name=f"Smoothed ER")

<Image layer 'SMOOTH' at 0x3b0d86860>

# ***CORE-PROCESSING prototype - masks_B***

## **`STEP 3` -  Semantic segmentation of the cytoplasm**

- apply MO thresholding method from the Allen Cell [aicssegmentation](https://github.com/AllenCell/aics-segmentation) package (threshold options = user input)

In [29]:
### USER INPUT REQUIRED ###
thresh_method = 'med'
cutoff_size =  200
thresh_adj = 0.4

bw_cyto = masked_object_thresh(structure_img_smooth, 
                          global_method=thresh_method, 
                          cutoff_size=cutoff_size, 
                          local_adjust=thresh_adj)

In [ ]:
viewer.add_image(bw_cyto, scale=scale, name=f"ER threshold")

<Image layer 'THRESHOLD' at 0x3efd40d60>

In [31]:
# FILL SMALL HOLES
hole_min_width = 0
hole_max_width = 40
# REMOVE SMALL OBJECTS
small_object_width = 15
fill_filter_method = "slice_by_slice"


cleaned_cyto = fill_and_filter_linear_size(bw_cyto, 
                                           hole_min=hole_min_width, 
                                           hole_max=hole_max_width, 
                                           min_size= small_object_width,
                                           method=fill_filter_method)

In [ ]:
viewer.add_image(cleaned_cyto, scale=scale, name=f"Cleaned ER threshold")

<Image layer 'cleaned' at 0x3b4620880>

## **`STEP 4` - Segmentation of nuclei ‘seeds’ from cytoplasm**

- binary opening of inverted aggregate cytoplasm mask (erosion -> dilation)

In [33]:
cytoplasm_inverse = 1 - cleaned_cyto

cytoplasm_inv_opened = binary_opening(cytoplasm_inverse, footprint=np.ones([3,3,3]))

In [ ]:
viewer.add_image(cytoplasm_inverse, scale=scale, name=f"Inverse ER")

<Image layer 'INVERSE' at 0x3b4376050>

In [35]:
max_nuc_width = 200

nuc_removed = fill_and_filter_linear_size(cytoplasm_inv_opened, 
                                          hole_max=0, 
                                          hole_min=0, 
                                          min_size=max_nuc_width, 
                                          method='3D')

viewer.add_image(nuc_removed, scale=scale)

<Image layer 'nuc_removed' at 0x3b40c4550>

In [36]:
nuc_objs = np.logical_xor(cytoplasm_inv_opened, nuc_removed)

In [ ]:
viewer.add_image(nuc_objs, scale=scale, name=f"Nuclei objects")

<Image layer 'nuclei objects' at 0x3b6ff2c50>

In [38]:
hole_max = 15
hole_min = 0
min_size = 50

nuc_cleaned = fill_and_filter_linear_size(nuc_objs, 
                                          hole_max=hole_max, 
                                          hole_min=hole_min, 
                                          min_size=min_size, 
                                          method='3D')

In [ ]:
viewer.add_image(nuc_cleaned, scale=scale, name=f"Nuclei cleaned")

<Image layer 'nuclei cleaned' at 0x3b70b6260>

In [40]:
# create instance segmentation based on connectivity
nuc_labels = label(nuc_cleaned).astype(np.uint16)

In [ ]:
viewer.add_labels(nuc_labels, scale=scale, name=f"Nuclei labels")

<Labels layer 'nuclei labels' at 0x3ef64cdc0>

## **`STEP 5` -  Filter and dilate nuclei**

In [ ]:
# Remove labels touching the image borders
nuc_labels_filter = clear_border(nuc_labels)

# Update the viewer with the filtered labels
viewer.add_labels(nuc_labels_filter, scale=scale, name="Nuclei labels filtered")

<Labels layer 'nuclei labels filtered' at 0x3b70b7010>

In [43]:
# Dilate by a few voxels (adjust the footprint as needed)
dilated_nuc = binary_dilation(nuc_labels_filter, footprint=np.ones((13,13,13)))  # 6x6x6 dilation

In [ ]:
viewer.add_labels(dilated_nuc, scale=scale, name=f"Dilated nuclei")

<Labels layer 'dilated nuclei' at 0x3639f9300>

## **`STEP 6` -  Segment nuclear envelope**

In [45]:
nuclear_envelope = np.logical_and(dilated_nuc, cleaned_cyto)

In [46]:
viewer.add_image(nuclear_envelope, scale=scale)

<Image layer 'nuclear_envelope' at 0x3645da680>

In [47]:
# Erode by a few voxels (adjust the footprint as needed)
eroded_nuc = binary_erosion(nuclear_envelope, footprint=np.ones((2,2,2)))

In [48]:
viewer.add_image(eroded_nuc, scale=scale, name="Eroded Nuclear Envelope")

<Image layer 'Eroded Nuclear Envelope' at 0x364885330>

In [49]:
## might need to add a hole filling step here ##

In [50]:
# ### OPTIONAL: THINNING STEP ###
# from aicssegmentation.core.utils import topology_preserving_thinning
# #### USER INPUT REQUIRED ###
# thin_dist_preserve = 4
# thin_dist = 2

In [51]:
# # thin segmentation with maintain a minimal required thickness
# ne_thin = topology_preserving_thinning(nuclear_envelope, thin_dist_preserve, thin_dist)

# # adding image to Napari as a new layer
# viewer.add_image(ne_thin, scale=scale, name="4 - Thinning")

# ***EXPORT***

## **`STEP 7` -  Export nuclear envelope label**

In [52]:
# create instance segmentation based on connectivity
env_labels = label_uint16(eroded_nuc)

# adding image to Napari as a new layer
viewer.add_labels(env_labels, scale=scale, name="Instance segmentation")

<Labels layer 'Instance segmentation' at 0x364c02320>

In [53]:
out_file_n = export_inferred_organelle(env_labels, "nucenv", meta_dict, out_data_path)
print(f"saved to: {out_data_path}")

saved file: WT_p17_ALLwell2_img12_Unmixing_cmle.ome-nucenv
saved to: /Volumes/Meredith/Meredith Microscopy data/SNCA iPSC MSi/Exp1_8_29_25/106_ne_test
